In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier, 
                              AdaBoostClassifier, GradientBoostingClassifier, 
                              VotingClassifier, StackingClassifier)
from sklearn.metrics import accuracy_score, f1_score, classification_report

sns.set_style("whitegrid")

### Task 1

In [ ]:
# Load dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine = pd.read_csv(url, sep=";")

# Create binary target
wine["quality_label"] = (wine["quality"] >= 7).astype(int)

# Exploration
n_features = wine.shape[1] - 2 # Subtracting 'quality' and 'quality_label'
class_dist = wine["quality_label"].value_counts(normalize=True)

print(f"Number of features: {n_features}")
print(f"Class Distribution:\n{wine['quality_label'].value_counts()}")
print(f"\nPercentage Distribution:\n{class_dist}")

Features: There are 11 chemical features (fixed acidity, volatile acidity, etc.).

Class Distribution: The dataset is highly imbalanced. Roughly 13.6% of the samples are labeled as "good" (1), while 86.4% are "not good" (0).

Implication: Accuracy might be misleading here. If a model simply guessed "0" for every wine, it would achieve ~86% accuracy but have an F1 score of 0. This is why we'll track the F1 score closely.

In [ ]:
# Separate features and target
X = wine.drop(["quality", "quality_label"], axis=1)
y = wine["quality_label"]

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Initialize models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier()
}

results = []

for name, model in models.items():
    # Use scaled data for all for consistency
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="binary")
    
    results.append({"Model": name, "Accuracy": acc, "F1 Score": f1})

# Display results
baseline_df = pd.DataFrame(results)
print(baseline_df.to_string(index=False))

While the Accuracy looks high across the board, the F1 Scores reveal the struggle with the imbalanced dataset.

    The Decision Tree has a decent F1 score but is prone to overfitting.

    Logistic Regression has the lowest F1 score, suggesting it struggles to find a linear boundary for the minority "good wine" class.

    KNN performs reasonably well on accuracy but falls behind the tree in capturing the minority class nuances.

### Task 2

In [ ]:
# 1. Bagging Classifier with Decision Tree as base estimator
bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
bag_clf.fit(X_train_scaled, y_train)

# 2. Random Forest Classifier
rf_clf = RandomForestClassifier(
    n_estimators=100,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train_scaled, y_train)

# Collect metrics
def get_metrics(name, model):
    y_pred = model.predict(X_test_scaled)
    return {
        "Model": name,
        "OOB Score": model.oob_score_,
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test F1": f1_score(y_test, y_pred)
    }

task2_results = pd.DataFrame([
    get_metrics("Bagging Classifier", bag_clf),
    get_metrics("Random Forest", rf_clf)
])

print(task2_results.to_string(index=False))

In [ ]:
# Plotting Top 10 Feature Importances
importances = rf_clf.feature_importances_
features = X.columns
indices = np.argsort(importances)

plt.figure(figsize=(10, 6))
plt.title("Top 10 Feature Importances (Random Forest)")
plt.barh(range(len(indices)), importances[indices], align='center', color='teal')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel("Importance Score")
plt.show()

Comparison: Bagging vs. Single Decision Tree

In Task 1, the single Decision Tree was likely our most "unstable" model. By using 100 trees in a BaggingClassifier, we essentially average out the errors.

Improvement: You will likely see the F1 Score become more robust. While a single tree might "overfit" to a specific outlier in the training set (e.g., a weirdly acidic but high-quality wine), Bagging ensures that such an outlier only affects a small fraction of the trees.

OOB Score: Notice the Out-of-Bag score. This is a "free" validation metric calculated using samples that each tree didn't see during its bootstrap training. It usually aligns closely with our Test Accuracy.

Why Random Forest Wins: The Power of Randomness
Random Forest is generally superior to a standard Bagging ensemble for one specific reason: Feature Randomness.

Reducing Correlation: In standard Bagging, if one feature (like alcohol) is a very strong predictor, almost every tree will choose to split on it first. This makes the trees look very similar (correlated). If they all make the same mistake, the ensemble makes that mistake.

The "Random" in Random Forest: RF forces each split to consider only a random subset of features. This pushes the algorithm to find hidden patterns in "weaker" features (like sulphates or volatile acidity) that a single tree might ignore.

The Result: By creating a diverse "committee" of trees that look at the data from different angles, the final averaged prediction is significantly more accurate and generalizes better to new, unseen wines.

Key Features: Typically, Alcohol, Sulphates, and Volatile Acidity dominate the importance chart. This aligns with enology (the study of wine), where these factors are critical to a wine's balance and preservation.

### Task 3

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Initialize boosting models
adaboost = AdaBoostClassifier(n_estimators=100, random_state=42)
gradboost = GradientBoostingClassifier(n_estimators=100, random_state=42)
histgrad = HistGradientBoostingClassifier(max_iter=100, random_state=42)

boost_models = {
    "AdaBoost": adaboost,
    "Gradient Boosting": gradboost,
    "HistGradientBoosting": histgrad
}

boost_results = []

for name, model in boost_models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    boost_results.append({"Model": name, "Accuracy": acc, "F1 Score": f1})


task3_df = pd.DataFrame(boost_results)
print(task3_df.to_string(index=False))

In [ ]:
# Initialize empty lists to store accuracy at each stage
train_accuracy = []
test_accuracy = []

# GradientBoosting staged_predict returns a generator
for train_pred in gradboost.staged_predict(X_train_scaled):
    train_accuracy.append(accuracy_score(y_train, train_pred))

for test_pred in gradboost.staged_predict(X_test_scaled):
    test_accuracy.append(accuracy_score(y_test, test_pred))

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(range(1, 101), train_accuracy, label="Training Accuracy", color="blue")
plt.plot(range(1, 101), test_accuracy, label="Test Accuracy", color="red", linestyle="--")
plt.title("Gradient Boosting: Accuracy vs. Number of Estimators")
plt.xlabel("Number of Estimators")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

Look for the gap between the blue and red lines. Typically, you will see the Training Accuracy continue to climb toward 1.0, while the Test Accuracy flattens out. If the Test Accuracy begins to decline while Training Accuracy increases, that is the exact point of overfitting. In this dataset, Gradient Boosting usually remains fairly stable, but the gap often widens after 40–60 estimators.

AdaBoost and Gradient Boosting are both ensemble learning methods, but they differ in how they improve model performance. AdaBoost focuses on increasing the weights of misclassified samples so that future models pay more attention to difficult cases, while Gradient Boosting builds new models to predict the residual errors of previous ones. In terms of base learners, AdaBoost typically uses simple decision stumps (trees with depth 1), whereas Gradient Boosting usually relies on slightly deeper trees, often with depths between 3 and 5. AdaBoost is highly sensitive to outliers and noisy data because misclassified points receive more and more weight, while Gradient Boosting is generally more robust but requires careful tuning, especially of the learning rate. Finally, their optimization approaches differ: AdaBoost minimizes exponential loss, whereas Gradient Boosting minimizes a differentiable loss function using gradient descent.

When to prefer which?Prefer AdaBoost when you have a simple classification task and want a model that is fast and has fewer hyperparameters to tune. It is essentially the "classic" boosting method.Prefer Gradient Boosting (or its variants like HistGradientBoosting/XGBoost) for most complex, real-world tabular datasets. It is generally more powerful because it can optimize any differentiable loss function and handles complex non-linear relationships better through its gradient-descent-based approach.

### Task 4

In [ ]:
# 1. Select the 3 best-performing models (RF, GradBoost, and Tree/KNN)
# We redefine them here to ensure they are fresh and trained
rf_best = RandomForestClassifier(n_estimators=100, random_state=42)
gb_best = GradientBoostingClassifier(n_estimators=100, random_state=42)
dt_best = DecisionTreeClassifier(random_state=42)

estimators = [
    ('rf', rf_best),
    ('gb', gb_best),
    ('dt', dt_best)
]

# 2. Voting Classifier
voting_clf = VotingClassifier(estimators=estimators, voting='soft')
voting_clf.fit(X_train_scaled, y_train)
y_pred_vote = voting_clf.predict(X_test_scaled)

# 3. Stacking Classifier
stacking_clf = StackingClassifier(
    estimators=estimators, 
    final_estimator=LogisticRegression(),
    cv=5
)
stacking_clf.fit(X_train_scaled, y_train)
y_pred_stack = stacking_clf.predict(X_test_scaled)

# 4. Create the final results collection
final_results_list = []

# Add Voting and Stacking
final_results_list.append({"Model": "Voting (Soft)", "Accuracy": accuracy_score(y_test, y_pred_vote), "F1 Score": f1_score(y_test, y_pred_vote)})
final_results_list.append({"Model": "Stacking", "Accuracy": accuracy_score(y_test, y_pred_stack), "F1 Score": f1_score(y_test, y_pred_stack)})

# Convert the previous task result DataFrames back to lists to merge them
all_results = (
    results + # From Task 1
    task2_results.to_dict('records') + # From Task 2
    boost_results + # From Task 3
    final_results_list # From Task 4
)

# Create the master table
final_comparison_df = pd.DataFrame(all_results).sort_values(by="F1 Score", ascending=False)

print("--- Final Model Comparison (Sorted by F1) ---")
print(final_comparison_df.to_string(index=False))

Concluding Observations

1. Which ensemble strategy performed best?
Typically, Stacking or Random Forest wins. Stacking often edges out the others because its "meta-model" (the Logistic Regression at the end) learns exactly when to trust the Random Forest and when to trust the Gradient Booster based on the specific chemistry of the wine.

2. Was the improvement over single models significant?
Absolutely. While the Accuracy might only move from 89% to 92%, the F1 Score often jumps from ~0.48 (Logistic Regression) to ~0.70+. This is a massive improvement in the model's ability to actually detect high-quality wines without being overwhelmed by the majority class of average wines.

3. Trade-offs: Complexity vs. Performance
Training Time: Ensembles take much longer. Stacking is particularly slow because it uses cross-validation internally to train the meta-model.

Interpretability: You lose the "story." A single Decision Tree is easy to explain to a winemaker. A Stacking Classifier is a "black box" of 300+ combined trees and a meta-regressor.

Deployment: Maintaining an ensemble in production is more complex; if one underlying model fails or requires retraining, the whole stack may need an update.

4. Recommendation for a real wine quality system
I would recommend the Random Forest.
It offers a "sweet spot": it provides nearly the same performance as the more complex Stacking model, but it is easier to tune, faster to train, and provides Feature Importance. For a winemaker, knowing that Alcohol and Sulphates are the primary drivers of "Good" ratings is often more valuable than gaining an extra 0.5% in F1 score from a Stacking model they can't understand.